In [3]:
import pandas as pd
import pickle

# Load dataset
df = pd.read_csv("C:/Users/Lekas/Downloads/PredSkill2/data/job_posting.csv")

# Clean
df = df.dropna()

# Normalize skills
df['skills'] = df['skills'].str.lower()
df['skills'] = df['skills'].apply(lambda x: [s.strip() for s in x.split(",")])

# Convert to text
df['skills_text'] = df['skills'].apply(lambda x: " ".join(x))

# 🔥 ADD DESCRIPTION (IMPORTANT)
df['text'] = df['skills_text'] + " " + df['job_description'].str.lower()

# -----------------------------
# Train Career Model
# -----------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['text'])
y = df['job_title']

career_model = LogisticRegression(max_iter=1000)
career_model.fit(X, y)

# Save
pickle.dump(career_model, open("career_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))

# -----------------------------
# Domain Detection
# -----------------------------
def detect_domain(title):
    title = title.lower()

    if "data" in title or "analyst" in title:
        return "Data Science"
    elif "machine learning" in title or "ai" in title:
        return "AI / ML"
    elif "web" in title or "frontend" in title or "backend" in title:
        return "Web Development"
    elif "mobile" in title or "android" in title or "ios" in title:
        return "Mobile Development"
    elif "cloud" in title or "devops" in title:
        return "Cloud / DevOps"
    else:
        return "Software Development"

df["domain"] = df["job_title"].apply(detect_domain)

# Train domain model
domain_model = LogisticRegression(max_iter=1000)
domain_model.fit(X, df["domain"])

# Save
pickle.dump(domain_model, open("domain_model.pkl","wb"))